# Notebook 08 — Selection-only context search (Gate 5)

Beam search over rows using only activation/retrieval objectives; reproducible, resumable, diversity-constrained, with matched nulls. Fast path uses a synthetic additive score.

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="08", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220701_smoke_eagle_08_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
reproducible + resumable beam search; matched random null for comparison.

In [4]:
from subliminal_icl import beam_search as bs
rng = np.random.default_rng(5)
pool = {f"r{i}": float(rng.normal(0,1)) for i in range(24)}
src = {r: f"s{int(r[1:])%6}" for r in pool}
score = lambda items: float(sum(pool[i] for i in items))
propose = lambda beam: list(pool.keys())
cfg = bs.SearchConfig(beam_width=6, proposal_count=24, max_steps=2)
cons = bs.DiversityConstraints(max_from_one_source=3)
st1 = bs.beam_search(score, propose, cfg, constraints=cons, source_of=src)
st2 = bs.beam_search(score, propose, cfg, constraints=cons, source_of=src)
repro = [b.items for b in st1.beams] == [b.items for b in st2.beams]
best = bs.best_beam(st1)
null = bs.best_beam(bs.beam_search(lambda it: float(sum(rng.normal(0,1) for _ in it)), propose, cfg, constraints=cons, source_of=src))
print("selected:", best.items, "score:", round(best.score,3), "reproducible:", repro, "null best:", round(null.score,3))

selected: ['r9', 'r13'] score: 3.235 reproducible: True null best: 3.49


## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("reproducible", repro, "same seed -> same beams"),
          ("diversity_respected", len(set(best.items)) == len(best.items), "no duplicate rows")]
gs = run_gate_checks("gate_05_selection_reconstruction", "Selection beats matched null", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,reproducible,PASS,same seed -> same beams
1,diversity_respected,PASS,no duplicate rows


GATE gate_05_selection_reconstruction -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.